# Bidirectional RNN - PyTorch


In [1]:
import os
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------------
# 1. Reproducibility & Device
# -----------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------------
# 2. Synthetic Dataset
# -----------------------------------
samples, seq_len, features = 1400, 30, 3
X = np.random.normal(size=(samples, seq_len, features)).astype("float32")

signal = (
    X[:, -10:, 0].mean(axis=1) +
    0.5 * X[:, :10, 1].mean(axis=1)
)
y = (signal > 0.05).astype("int64")

train_loader = DataLoader(
    TensorDataset(torch.tensor(X[:1000]), torch.tensor(y[:1000])),
    batch_size=64,
    shuffle=True
)

test_x = torch.tensor(X[1000:], device=device)
test_y = torch.tensor(y[1000:], device=device)

# -----------------------------------
# 3. Model Definition
# -----------------------------------
class BiRNNClassifier(nn.Module):
    def __init__(self, input_dim: int = 3, hidden_dim: int = 32):
        super().__init__()
        self.recurrent = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(hidden_dim * 2, 2)

    def forward(self, x):
        output, _ = self.recurrent(x)
        return self.classifier(output[:, -1, :])

# -----------------------------------
# 4. Initialize Model, Optimizer, Loss
# -----------------------------------
model = BiRNNClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# -----------------------------------
# 5. Training & Evaluation
# -----------------------------------
for epoch in range(10):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        accuracy = (model(test_x).argmax(dim=1) == test_y).float().mean().item()

    print(f"Epoch {epoch+1:02d} | Accuracy: {accuracy:.3f}")

Using device: cuda
Epoch 01 | Accuracy: 0.620
Epoch 02 | Accuracy: 0.647
Epoch 03 | Accuracy: 0.767
Epoch 04 | Accuracy: 0.803
Epoch 05 | Accuracy: 0.817
Epoch 06 | Accuracy: 0.825
Epoch 07 | Accuracy: 0.832
Epoch 08 | Accuracy: 0.837
Epoch 09 | Accuracy: 0.837
Epoch 10 | Accuracy: 0.817


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# -----------------------------------
# 1. Device Configuration
# -----------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------
# 2. Hyperparameters
# -----------------------------------
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBED_DIM = 128
HIDDEN_SIZE = 64
NUM_CLASSES = 1
BATCH_SIZE = 64
EPOCHS = 5

# -----------------------------------
# 3. Load Dataset
# -----------------------------------
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# Padding sequences
x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding="post")
x_test = pad_sequences(x_test, maxlen=MAX_LEN, padding="post")

# Convert to tensors
x_train = torch.tensor(x_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.float32)

# -----------------------------------
# 4. DataLoaders
# -----------------------------------
train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# -----------------------------------
# 5. Bidirectional RNN Model
# -----------------------------------
class BiRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, output_size):
        super(BiRNN, self).__init__()

        # Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Bidirectional RNN
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )

        # Fully Connected Layers
        self.fc1 = nn.Linear(hidden_size * 2, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Embedding
        embedded = self.embedding(x)

        # RNN Output
        output, hidden = self.rnn(embedded)

        # Forward and backward hidden states
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        # Concatenate both directions
        hidden_cat = torch.cat((forward_hidden, backward_hidden), dim=1)

        # Fully connected layers
        x = self.fc1(hidden_cat)
        x = self.relu(x)
        x = self.fc2(x)

        return self.sigmoid(x)

# -----------------------------------
# 6. Initialize Model
# -----------------------------------
model = BiRNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    output_size=NUM_CLASSES
).to(device)

# -----------------------------------
# 7. Loss and Optimizer
# -----------------------------------
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -----------------------------------
# 8. Training Loop
# -----------------------------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.unsqueeze(1)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

# -----------------------------------
# 9. Evaluation
# -----------------------------------
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        predictions = (outputs >= 0.5).float()
        total += labels.size(0)
        correct += (predictions.squeeze() == labels).sum().item()

accuracy = 100 * correct / total
print(f"\nTest Accuracy: {accuracy:.2f}%")

# -----------------------------------
# 10. Prediction Example
# -----------------------------------
sample = x_test[0].unsqueeze(0).to(device)
prediction = model(sample)

print("Positive Review" if prediction.item() > 0.5 else "Negative Review")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch [1/5], Loss: 0.6846
Epoch [2/5], Loss: 0.6503
Epoch [3/5], Loss: 0.6209
Epoch [4/5], Loss: 0.5870
Epoch [5/5], Loss: 0.5318

Test Accuracy: 67.35%
Positive Review
